# Unit 12 — Graph Theory Solutions

Worked answers for the short-answer exercises, and reference programs for the programming exercises.
Each programming solution is also saved as a runnable program in the `assets` folder (for example `assets/ex17.py`); run it with `python assets/ex17.py < input.txt`.
Each short-answer check cell uses a small graph helper kept in `assets/verify/graph_eval.py`, which lists paths and cycles and multiplies adjacency matrices, to confirm the worked answer.
Exercises 1 to 9 count as ACSL's Elementary handout does (simple paths; a cycle once in each direction); the later exercises count as ACSL's wiki does (a path of length p may repeat vertices; a cycle counts once).

## Exercise 1

First write down each vertex's neighbours: `A: B C D`, `B: A C`, `C: A B D`, `D: A C E`, `E: D`.
Then grow every path from `A`, one edge at a time, never going back to a vertex already on the path.

- `A → B → C →` `D` gives `ABCD` (`C → A` is not allowed, `A` is used).
- `A → C → B →` nothing new (`B`'s neighbours `A` and `C` are both used).
- `A → C → D →` `E` gives `ACDE`.
- `A → D → C →` `B` gives `ADCB`.
- `A → D → E →` nothing new (`E`'s only neighbour is `D`).

In alphabetical order:

**Answer:** `ABCD, ACDE, ADCB`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

edges = "{AB, AC, AD, BC, CD, DE}"
paths = graph_eval.simple_paths(edges, "A", length=3, vertices="ABCDE")
assert str(", ".join(paths)) == "ABCD, ACDE, ADCB"

## Exercise 2

Find each different loop first, then count it twice (once in each direction).

`A` has only the neighbours `B` and `C`, so a loop through `A` goes `A → B → … → C → A`, and the middle part is a way from `B` to `C` that avoids `A`:

- `B → C` gives the triangle `ABCA`
- `B → D → C` gives `ABDCA`
- `B → D → E → C` gives `ABDECA`

Loops that avoid `A` use only `B, C, D, E`, with edges `BC, BD, CD, CE, DE` (every pair except `B` and `E`):

- the triangles `BCDB` and `CDEC`
- the square `B → C → E → D → B`, that is `BCEDB` (the other two ways round a square need the missing edge `BE`)

That is 6 different loops, and each one counts once in each direction: 6 × 2 = 12.

**Answer:** `12`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

edges = "{AB, AC, BC, BD, CD, CE, DE}"
both_ways = graph_eval.cycles(edges, both_directions=True, vertices="ABCDE")
assert len(graph_eval.cycles(edges, vertices="ABCDE")) == 6
assert str(len(both_ways)) == "12"

## Exercise 3

The degree of `C` is the number of edges that touch `C`: `AC`, `BC`, `CD`, `CE` and `CF`.

**Answer:** `5`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

edges = "{AB, AC, AF, BC, CD, CE, CF, DE}"
assert str(graph_eval.degrees(edges, vertices="ABCDEF")["C"]) == "5"

## Exercise 4

Count each vertex's degree:

| Vertex | Edges | Degree |
|--------|-------|--------|
| A | AB, AE | 2 |
| B | AB, BC, BD | 3 |
| C | BC, CD | 2 |
| D | BD, CD, DE | 3 |
| E | AE, DE | 2 |

The graph is in one piece, and exactly two vertices (`B` and `D`) have odd degree, so it is traversable, starting at `B` and ending at `D` (for example `B → A → E → D → B → C → D`).

**Answer:** `YES`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

edges = "{AB, AE, BC, BD, CD, DE}"
assert str("YES" if graph_eval.traversable(edges, vertices="ABCDE") else "NO") == "YES"

## Exercise 5

In a complete graph every vertex is joined to each of the other 8.
That makes 9 × 8 = 72 edge ends, and each edge has two ends, so there are 72 ÷ 2 = 36 edges.

**Answer:** `36`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

from itertools import combinations

letters = "ABCDEFGHI"
complete = " ".join(a + b for a, b in combinations(letters, 2))
edge_count = sum(graph_eval.degrees(complete).values()) // 2
assert str(edge_count) == "36"

## Exercise 6

Neighbours: `A: B D`, `B: A C D`, `C: B E`, `D: A B E`, `E: C D`.
Start at `B` and try each first step, stopping whenever the path reaches `E`:

- `B → A → D → E`: `BADE`
- `B → C → E`: `BCE`
- `B → D → E`: `BDE`; `B → D → A` is stuck (`A`'s neighbours `B` and `D` are used)

In alphabetical order:

**Answer:** `BADE, BCE, BDE`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

edges = "{AB, AD, BC, BD, CE, DE}"
paths = graph_eval.simple_paths(edges, "B", end="E", vertices="ABCDE")
assert str(", ".join(paths)) == "BADE, BCE, BDE"

## Exercise 7

A simple path of length 2 is `X → M → Y`: a middle vertex `M` and two different neighbours of it, in order.
A vertex of degree `d` is the middle of `d × (d − 1)` such paths.

| Middle | Neighbours | Paths |
|--------|------------|-------|
| A | B | 1 × 0 = 0 |
| B | A, C, D | 3 × 2 = 6 (`ABC`, `ABD`, `CBA`, `CBD`, `DBA`, `DBC`) |
| C | B, D | 2 × 1 = 2 (`BCD`, `DCB`) |
| D | B, C, E | 3 × 2 = 6 (`BDC`, `BDE`, `CDB`, `CDE`, `EDB`, `EDC`) |
| E | D | 1 × 0 = 0 |

Total: 0 + 6 + 2 + 6 + 0 = 14.

**Answer:** `14`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

edges = "{AB, BC, BD, CD, DE}"
count = 0
for v in "ABCDE":
    count = count + len(graph_eval.simple_paths(edges, v, length=2, vertices="ABCDE"))
assert str(count) == "14"

## Exercise 8

The loops through `C` are:

- the triangle `A, B, C`: `CABC` and `CBAC`
- the triangle `B, C, D`: `CBDC` and `CDBC`
- the square `C → A → B → D → C` (edges `CA`, `AB`, `BD`, `DC`): `CABDC` and `CDBAC`

In alphabetical order:

**Answer:** `CABC, CABDC, CBAC, CBDC, CDBAC, CDBC`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

edges = "{AB, AC, BC, BD, CD}"
loops = graph_eval.cycles(edges, start="C", vertices="ABCD")
assert str(", ".join(loops)) == "CABC, CABDC, CBAC, CBDC, CDBAC, CDBC"

## Exercise 9

Every vertex has degree 2, so no vertex has odd degree.
But the graph is in two separate pieces, the triangle `A, B, C` and the triangle `D, E, F`, and a pencil that never lifts cannot get from one to the other.
So the graph is not traversable.

**Answer:** `NO`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

edges = "{AB, AC, BC, DE, DF, EF}"
assert graph_eval.components(edges, vertices="ABCDEF") == 2
assert str("YES" if graph_eval.traversable(edges, vertices="ABCDEF") else "NO") == "NO"

## Exercise 10

Out-going edges: `A: B`, `B: C`, `C: A D`, `D: C E`, `E: A B`.
Look for cycles starting at each vertex, following the arrows, and write each one from its smallest vertex so that none is counted twice.

- From `A` (only to `B`, then only to `C`): `C → A` gives `ABCA`; `C → D → E → A` gives `ABCDEA` (`C → D → E → B` repeats `B`).
- From `B`, avoiding `A`: `B → C → D → E → B` gives `BCDEB`.
- From `C`, avoiding `A` and `B`: `C → D → C` gives `CDC`.
- From `D` or `E`, avoiding all smaller letters: `D → E` then `E` only goes to `A` or `B`, so nothing new.

The cycles are `ABCA`, `ABCDEA`, `BCDEB` and `CDC`.

**Answer:** `4`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

edges = "{AB, BC, CA, CD, DC, DE, EB, EA}"
loops = graph_eval.cycles(edges, vertices="ABCDE", directed=True)
assert loops == ["ABCA", "ABCDEA", "BCDEB", "CDC"]
assert str(len(loops)) == "4"

## Exercise 11

The edges `AB`, `BC`, `CD` form a chain `A - B - C - D`, and `E` is joined to all four of them.
The chain by itself has no loop, so every cycle goes through `E`: it leaves `E`, runs along a stretch of the chain, and comes back to `E`.
A stretch is fixed by its two end vertices on the chain, and there are 6 ways to choose them:

- `A…B`: `ABEA`; `B…C`: `BCEB`; `C…D`: `CDEC`
- `A…C`: `ABCEA`; `B…D`: `BCDEB`
- `A…D`: `ABCDEA`

Each cycle counts once, so the answer is 6.

**Answer:** `6`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

edges = "{AB, AE, BC, BE, CD, CE, DE}"
loops = graph_eval.cycles(edges, vertices="ABCDE")
assert loops == ["ABCDEA", "ABCEA", "ABEA", "BCDEB", "BCEB", "CDEC"]
assert str(len(loops)) == "6"

## Exercise 12

A path of length 2 from `A` to `D` is `A → X → D`.
`A` goes to `B` and `C`; `B → D` and `C → D` are both edges.
So the paths are `ABD` and `ACD`.

With the matrix, entry (A, D) of `M²` is row `A` of `M` times column `D` of `M`:

| | A | B | C | D |
|---|---|---|---|---|
| row A | 0 | 1 | 1 | 0 |
| column D | 0 | 1 | 1 | 0 |

0·0 + 1·1 + 1·1 + 0·0 = 2.

**Answer:** `2`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

edges = "{AB, AC, BC, BD, CA, CD, DA}"
assert str(graph_eval.count_paths(edges, "A", "D", 2, vertices="ABCD", directed=True)) == "2"

## Exercise 13

From the matrix the edges are `AB`, `AD`, `BC`, `CA`, `CD` and `DB`.
List the paths `C → X → Y → C`:

- `C → A → B → C`
- `C → A → D →` only `B`, not `C`
- `C → D → B → C`

That is 2 paths.

With the matrix: row `C` of `M²` is `A`'s row plus `D`'s row, `(0,1,0,1) + (0,1,0,0) = (0,2,0,1)`.
Entry (C, C) of `M³` is that row times column `C` of `M`, `(0,1,0,0)`: 0 + 2 + 0 + 0 = 2.

**Answer:** `2`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

edges = "AB AD BC CA CD DB"
m = [[0, 1, 0, 1], [0, 0, 1, 0], [1, 0, 0, 1], [0, 1, 0, 0]]
assert graph_eval.matrix(edges, directed=True) == m
assert graph_eval.matrix_power(m, 3)[2][2] == graph_eval.count_paths(edges, "C", "C", 3, directed=True)
assert str(graph_eval.matrix_power(m, 3)[2][2]) == "2"

## Exercise 14

The first step from `B` goes to one of its neighbours `A`, `C` or `D`, and the second step can use any edge of that neighbour, including the one back to `B`:

- from `A` (neighbours `B`, `C`): `BAB`, `BAC`
- from `C` (neighbours `A`, `B`): `BCA`, `BCB`
- from `D` (neighbours `B`, `E`): `BDB`, `BDE`

That is 2 + 2 + 2 = 6, the sum of row `B` of `M²`.

**Answer:** `6`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

edges = "{AB, AC, BC, BD, DE}"
total = 0
for end in "ABCDE":
    total = total + graph_eval.count_paths(edges, "B", end, 2, vertices="ABCDE")
assert str(total) == "6"

## Exercise 15

Group the vertices that are joined:

- `AE` and `CE`: `{A, C, E}`
- `BG` and `GH`: `{B, G, H}`
- `DF`: `{D, F}`
- `I` has no edge: `{I}` on its own

That is 4 connected components.

**Answer:** `4`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

edges = "{AE, BG, CE, DF, GH}"
assert str(graph_eval.components(edges, vertices="ABCDEFGHI")) == "4"

## Exercise 16

A tree with `N` vertices has `N − 1` edges: one fewer edge than vertices.
So in a forest each tree is "one edge short", and the number of trees is the number of vertices minus the number of edges: 15 − 11 = 4.

The check builds one such forest (a chain of 12 vertices, which has 11 edges, plus 3 lone vertices) and counts its pieces.

**Answer:** `4`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

forest = "AB BC CD DE EF FG GH HI IJ JK KL"
vertices = "ABCDEFGHIJKLMNO"
assert len(vertices) == 15
assert sum(graph_eval.degrees(forest, vertices=vertices).values()) // 2 == 11
assert graph_eval.cycles(forest, vertices=vertices) == []
assert graph_eval.components(forest, vertices=vertices) == 15 - 11
assert str(graph_eval.components(forest, vertices=vertices)) == "4"

## Exercise 17

**Notice:** the matrix starts as all zeros, and each edge sets one entry, or two for an undirected graph (`matrix[a][b]` and `matrix[b][a]`). The out-degree of a vertex is its row sum and the in-degree its column sum, so one pair of loops prints both. `spot` turns a letter into its position in the second line, so an edge written `DA` works too.

In [ ]:
def spot(names, letter):
    for i in range(len(names)):
        if names[i] == letter:
            return i
    return -1


kind = input()
names = input()
edges = input().split()

n = len(names)
matrix = []
for r in range(n):
    row = []
    for c in range(n):
        row.append(0)
    matrix.append(row)

for edge in edges:
    a = spot(names, edge[0])
    b = spot(names, edge[1])
    matrix[a][b] = 1
    if kind == "UNDIRECTED":
        matrix[b][a] = 1

for r in range(n):
    line = str(matrix[r][0])
    for c in range(1, n):
        line = line + " " + str(matrix[r][c])
    print(line)

for v in range(n):
    out_degree = 0
    in_degree = 0
    for k in range(n):
        out_degree = out_degree + matrix[v][k]
        in_degree = in_degree + matrix[k][v]
    print(names[v], out_degree, in_degree)

## Exercise 18

**Notice:** `multiply` is matrix multiplication with three nested loops: entry `[i][j]` of the product is the sum of `x[i][k] * y[k][j]` over every `k`. `M²` is `M` times `M`, and `M³` is `M²` times `M`; the answers are the entries in row `X`, column `Y`. When `X` and `Y` are the same, the entry is on the diagonal.

In [ ]:
def spot(names, letter):
    for i in range(len(names)):
        if names[i] == letter:
            return i
    return -1


def zeros(n):
    grid = []
    for r in range(n):
        row = []
        for c in range(n):
            row.append(0)
        grid.append(row)
    return grid


def multiply(x, y):
    n = len(x)
    product = zeros(n)
    for i in range(n):
        for j in range(n):
            total = 0
            for k in range(n):
                total = total + x[i][k] * y[k][j]
            product[i][j] = total
    return product


names = input()
edges = input().split()
ends = input().split()

m = zeros(len(names))
for edge in edges:
    m[spot(names, edge[0])][spot(names, edge[1])] = 1

square = multiply(m, m)
cube = multiply(square, m)
x = spot(names, ends[0])
y = spot(names, ends[1])
print(square[x][y])
print(cube[x][y])

## Exercise 19

List the simple paths from `A` to `F`, working through `D` or `E`, the only neighbours of `F`.

Cheapest ways to reach `D`: `A → C → B → D` costs 2 + 1 + 5 = 8 (`A → B → D` costs 9, `A → C → D` costs 10).
Cheapest ways to reach `E`: `A → C → B → D → E` costs 8 + 2 = 10 (`A → C → E` costs 12).

| Path | Weight |
|------|--------|
| A C B D F | 2 + 1 + 5 + 6 = 14 |
| A C B D E F | 2 + 1 + 5 + 2 + 3 = 13 |
| A C E F | 2 + 10 + 3 = 15 |
| A B D E F | 4 + 5 + 2 + 3 = 14 |
| A C D E F | 2 + 8 + 2 + 3 = 15 |

Every other path costs more, so the smallest weight is 13, along `ACBDEF`.

**Answer:** `13`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

edges = "{AB4, AC2, BC1, BD5, CD8, CE10, DE2, DF6, EF3}"
assert str(graph_eval.cheapest(edges, "A", "F", vertices="ABCDEF")) == "13"

## Exercise 20

Look for a cycle in each option:

- **(A)** `A → B → C → A` is a cycle: not a DAG.
- **(B)** Arrows go `A → B → D` and `A → C → D`; nothing leaves `D`, so no way back: a DAG.
- **(C)** `B → C → D → B` is a cycle: not a DAG.
- **(D)** Arrows go `B → A → D` and `B → C → D`; nothing enters `B` and nothing leaves `D`: a DAG.
- **(E)** `A → B → A` is a cycle: not a DAG.

**Answer:** `B, D`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

options = {
    "A": "{AB, BC, CA}",
    "B": "{AB, AC, BD, CD}",
    "C": "{AB, BC, CD, DB}",
    "D": "{AD, BA, BC, CD}",
    "E": "{AB, BA}",
}
dags = [label for label in sorted(options) if graph_eval.cycles(options[label], directed=True) == []]
assert str(", ".join(dags) if dags else "NONE") == "B, D"

## Exercise 21

The graph is two triangles, `A, B, C` and `C, D, E`, that share only vertex `C`.
A spanning tree has 5 − 1 = 4 edges and no cycle, so it must drop exactly one edge from each triangle (dropping two edges of one triangle cuts a vertex off).

Each triangle has 3 edges to drop, so there are 3 × 3 = 9 spanning trees.

**Answer:** `9`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

from itertools import combinations

edges = ["AB", "AC", "BC", "CD", "CE", "DE"]
trees = 0
for chosen in combinations(edges, 4):
    if graph_eval.components(" ".join(chosen), vertices="ABCDE") == 1:
        trees = trees + 1
assert str(trees) == "9"

## Exercise 22

In the order `A, B, C, D`:

| M | A | B | C | D |
|---|---|---|---|---|
| **A** | 1 | 1 | 1 | 0 |
| **B** | 0 | 0 | 1 | 0 |
| **C** | 1 | 0 | 0 | 1 |
| **D** | 0 | 1 | 0 | 0 |

Only row `A` of each power is needed, and each new row is the old row times `M`.

- Row `A` of `M²` = 1·(row A) + 1·(row B) + 1·(row C) = `(1,1,1,0) + (0,0,1,0) + (1,0,0,1)` = `(2,1,2,1)`.
- Row `A` of `M³` = 2·(row A) + 1·(row B) + 2·(row C) + 1·(row D) = `(2,2,2,0) + (0,0,1,0) + (2,0,0,2) + (0,1,0,0)` = `(4,3,3,2)`.
- Row `A` of `M⁴` = 4·(row A) + 3·(row B) + 3·(row C) + 2·(row D) = `(4,4,4,0) + (0,0,3,0) + (3,0,0,3) + (0,2,0,0)` = `(7,6,7,3)`.

The `D` entries are 1, 2 and 3 (for example, length 2 is `ACD`; length 3 is `AACD` and `ABCD`), so the total is 1 + 2 + 3 = 6.

**Answer:** `6`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

edges = "{AA, AB, AC, BC, CA, CD, DB}"
counts = [graph_eval.count_paths(edges, "A", "D", p, vertices="ABCD", directed=True) for p in (2, 3, 4)]
assert counts == [1, 2, 3]
assert str(sum(counts)) == "6"

## Exercise 23

**Notice:** `cheapest` tries every simple path from `X` by recursion, marking a vertex visited on the way in and unmarking it on the way back, and keeps the smallest total that reaches `Y`. `-1` stands for "no path found", which turns into `NONE`. The weight is everything after the two letters, `int(edge[2:])`, so two-digit weights such as `CE12` work.

In [ ]:
def spot(names, letter):
    for i in range(len(names)):
        if names[i] == letter:
            return i
    return -1


def cheapest(here, goal, visited, cost, weight):
    if here == goal:
        return cost
    best = -1
    for nxt in range(len(weight)):
        if weight[here][nxt] > 0 and not visited[nxt]:
            visited[nxt] = True
            total = cheapest(nxt, goal, visited, cost + weight[here][nxt], weight)
            visited[nxt] = False
            if total != -1:
                if best == -1 or total < best:
                    best = total
    return best


names = input()
edges = input().split()
ends = input().split()

n = len(names)
weight = []
visited = []
for r in range(n):
    row = []
    for c in range(n):
        row.append(0)
    weight.append(row)
    visited.append(False)

for edge in edges:
    a = spot(names, edge[0])
    b = spot(names, edge[1])
    w = int(edge[2:])
    weight[a][b] = w
    weight[b][a] = w

start = spot(names, ends[0])
visited[start] = True
answer = cheapest(start, spot(names, ends[1]), visited, 0, weight)
if answer == -1:
    print("NONE")
else:
    print(answer)

## Exercise 24

Out-going edges: `A: B D`, `B: C`, `C: A D`, `D: A B`.
Follow every route from `A` that never repeats a vertex, and note each time an arrow leads back to `A`:

- `A → B → C → A`: `ABCA`
- `A → B → C → D → A`: `ABCDA` (`D → B` repeats `B`)
- `A → D → A`: `ADA`
- `A → D → B → C → A`: `ADBCA` (`C → D` repeats `D`)

In alphabetical order:

**Answer:** `ABCA, ABCDA, ADA, ADBCA`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

edges = "{AB, AD, BC, CA, CD, DA, DB}"
loops = graph_eval.cycles(edges, start="A", vertices="ABCD", directed=True)
assert str(", ".join(loops)) == "ABCA, ABCDA, ADA, ADBCA"

## Exercise 25

**Before:** `A` goes only to `B`, `B` only to `C`, and `C` to `A` or `D`: the paths of length 3 are `ABCA` and `ABCD`, so 2.

**After:** out-going edges are `A: B D`, `B: C`, `C: A D`, `D: A B`.
Count the paths step by step (how many ways to be at each vertex):

| Step | A | B | C | D |
|------|---|---|---|---|
| 0 | 1 | 0 | 0 | 0 |
| 1 | 0 | 1 | 0 | 1 |
| 2 | 1 | 1 | 1 | 0 |
| 3 | 1 | 1 | 1 | 2 |

(Step 2: `B → C`, `D → A`, `D → B`. Step 3: from `A` to `B` and `D`, from `B` to `C`, from `C` to `A` and `D`.)
That is 1 + 1 + 1 + 2 = 5 paths: `ABCA`, `ABCD`, `ADAB`, `ADAD`, `ADBC`.

5 − 2 = 3 more.

**Answer:** `3`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import graph_eval

before = "{AB, BC, CA, CD, DB}"
after = "{AB, BC, CA, CD, DB, AD, DA}"
count_before = sum(graph_eval.count_paths(before, "A", e, 3, vertices="ABCD", directed=True) for e in "ABCD")
count_after = sum(graph_eval.count_paths(after, "A", e, 3, vertices="ABCD", directed=True) for e in "ABCD")
assert (count_before, count_after) == (2, 5)
assert str(count_after - count_before) == "3"

## Exercise 26

**Notice:** each cycle is counted once by starting it only at its smallest vertex: `loops` never steps to a vertex with a smaller position than `first`. The search goes forward only along edges that exist, so a directed cycle is found only in its own direction, and a 2-cycle `ABA` is found because `A → B → A` returns to `first` after one step. The sample's cycles `ABA`, `BCDB` and `CDC` start at `A`, `B` and `C`.

In [ ]:
def spot(names, letter):
    for i in range(len(names)):
        if names[i] == letter:
            return i
    return -1


def loops(first, here, visited, matrix):
    found = 0
    for nxt in range(len(matrix)):
        if matrix[here][nxt] == 1:
            if nxt == first:
                found = found + 1
            elif nxt > first and not visited[nxt]:
                visited[nxt] = True
                found = found + loops(first, nxt, visited, matrix)
                visited[nxt] = False
    return found


names = input()
edges = input().split()

n = len(names)
matrix = []
visited = []
for r in range(n):
    row = []
    for c in range(n):
        row.append(0)
    matrix.append(row)
    visited.append(False)

for edge in edges:
    matrix[spot(names, edge[0])][spot(names, edge[1])] = 1

total = 0
for first in range(n):
    visited[first] = True
    total = total + loops(first, first, visited, matrix)
    visited[first] = False
print(total)